# Exceptions => Logging

`logging` records what a program does. Unlike `print()`, log messages have **levels**, can be filtered, and can be sent to files.

| Level | Value | Use it for |
|---|---|---|
| `DEBUG` | 10 | Detailed information for developers |
| `INFO` | 20 | Normal events (started, saved, finished) |
| `WARNING` | 30 | Something unexpected, the program still works. **Default threshold** |
| `ERROR` | 40 | An operation failed |
| `CRITICAL` | 50 | The program may not be able to continue |

| Tool | Purpose |
|---|---|
| `logging.getLogger(name)` | Get a logger |
| `logger.debug/info/warning/error/critical()` | Log at a level |
| `logger.exception()` | Log an error **with the traceback** (use inside `except`) |
| `logging.basicConfig()` | Quick setup for simple scripts |
| `Handler` | Where messages go (`StreamHandler`, `FileHandler`) |
| `Formatter` | How messages look |
| `logger.setLevel()` | Minimum level for a logger |

---

## Quick Start

```python
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logging.info("started")
```

* Without configuration, only `WARNING` and above are shown.
* `basicConfig()` does nothing if the root logger is already configured. In a notebook, use `force=True`.

---

## The Four Parts

| Part | Role |
|---|---|
| **Logger** | The object you call (`logger.info`) |
| **Level** | Messages below the level are dropped |
| **Handler** | Sends the message to a destination |
| **Formatter** | Builds the final text |

---

## Loggers Are Named and Hierarchical

```python
logger = logging.getLogger(__name__)      # one logger per module
```

* `"app.db"` is a **child** of `"app"`. Messages travel up to the parent's handlers (`propagate`).
* Configure handlers once, in the main program. Libraries should only create loggers.

---

## Useful Format Fields

| Field | Meaning |
|---|---|
| `%(asctime)s` | Time |
| `%(levelname)s` | Level name |
| `%(name)s` | Logger name |
| `%(message)s` | The message |
| `%(filename)s:%(lineno)d` | Where it was logged |

---

## Logging Errors

```python
try:
    risky()
except Exception:
    logger.exception("risky failed")      # ERROR + full traceback
```

---

## Lazy Formatting

Pass values as arguments, not through an f-string:

```python
logger.info("user=%s items=%d", user, count)
```

The text is only built if the message is actually logged.

---

## `logging` vs `print()`

| | `print()` | `logging` |
|---|---|---|
| Levels | No | Yes |
| Turn off without editing code | No | Yes |
| Send to a file | Manual | Built in |
| Time, module and line info | No | Yes |
| Best for | Quick checks, program output | Anything you keep |

## Source

https://docs.python.org/3/howto/logging.html

https://docs.python.org/3/library/logging.html

In [ ]:
import io
import logging
import tempfile
from pathlib import Path

# Levels are numbers
print(logging.DEBUG, logging.INFO, logging.WARNING, logging.ERROR, logging.CRITICAL)

# A logger that writes to an in-memory stream, so the output is easy to show
stream = io.StringIO()
handler = logging.StreamHandler(stream)
handler.setFormatter(logging.Formatter("%(levelname)s | %(name)s | %(message)s"))

logger = logging.getLogger("app")
logger.setLevel(logging.INFO)
logger.addHandler(handler)
logger.propagate = False                        # keep the demo output out of the root logger

logger.debug("hidden: below INFO")
logger.info("started")
logger.warning("disk almost full")
logger.error("save failed")
print(stream.getvalue())

# Child loggers send messages up to the parent's handlers
child = logging.getLogger("app.db")
child.info("connected")
print(stream.getvalue().splitlines()[-1])

# Change the level
logger.setLevel(logging.ERROR)
logger.info("now hidden")
logger.error("still shown")
print(stream.getvalue().splitlines()[-1])

# Lazy formatting: pass arguments instead of building the text yourself
logger.setLevel(logging.INFO)
logger.info("user=%s items=%d", "ann", 3)
print(stream.getvalue().splitlines()[-1])

# logger.exception() logs the traceback
try:
    1 / 0
except ZeroDivisionError:
    logger.exception("calculation failed")
lines = stream.getvalue().splitlines()
print(any(line.startswith("ERROR | app | calculation failed") for line in lines), lines[-1])

# A file handler
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "app.log"
    file_handler = logging.FileHandler(path, encoding="utf-8")
    file_handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    file_logger = logging.getLogger("app.files")
    file_logger.addHandler(file_handler)
    file_logger.propagate = False
    file_logger.warning("written to a file")
    file_handler.close()
    print(path.read_text(encoding="utf-8").strip())
    file_logger.removeHandler(file_handler)

# basicConfig on the root logger (force=True replaces existing handlers, e.g. in notebooks)
root_stream = io.StringIO()
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", stream=root_stream, force=True)
logging.info("via the root logger")
print(root_stream.getvalue().strip())

# Clean up the demo handlers and restore the root logger
logger.removeHandler(handler)
logging.getLogger().handlers.clear()
logging.getLogger().setLevel(logging.WARNING)